In [1]:
import pandas as pd
import numpy as np
import os
from datetime import datetime

np.random.seed(42)

# =========================================
# 1️⃣ CREAR CARPETA DE REPORTES
# =========================================
if not os.path.exists("reportes"):
    os.makedirs("reportes")

# =========================================
# 2️⃣ FUNCIONES PARA GENERAR BASES
# =========================================

def generar_rrhh(n=500):
    cargos = ["Analista", "Asistente", "Supervisor", "Gerente"]
    df = pd.DataFrame({
        "ID": range(1, n+1),
        "Cargo": np.random.choice(cargos, n, p=[0.5,0.3,0.15,0.05]),
        "Edad": np.random.randint(22, 60, n),
        "Salario": np.random.normal(3000, 800, n).round(2)
    })
    return df

def generar_ventas(n=500):
    df = pd.DataFrame({
        "ID_Venta": range(1, n+1),
        "Vendedor": np.random.randint(1,50,n),
        "Monto": np.random.gamma(2, 500, n).round(2),
        "Unidades": np.random.randint(1,20,n)
    })
    return df

def generar_finanzas(n=500):
    df = pd.DataFrame({
        "ID_Transaccion": range(1,n+1),
        "Ingreso": np.random.normal(10000,2000,n).round(2),
        "Costo": np.random.normal(7000,1500,n).round(2)
    })
    df["Utilidad"] = df["Ingreso"] - df["Costo"]
    return df

def generar_produccion(n=500):
    df = pd.DataFrame({
        "ID_Lote": range(1,n+1),
        "Unidades_Producidas": np.random.randint(100,1000,n),
        "Defectuosas": np.random.randint(0,50,n)
    })
    return df

def generar_marketing(n=500):
    df = pd.DataFrame({
        "Campaña_ID": range(1,n+1),
        "Inversion": np.random.normal(5000,1000,n).round(2),
        "Clicks": np.random.randint(100,5000,n),
        "Conversiones": np.random.randint(10,500,n)
    })
    return df


# =========================================
# 3️⃣ GENERAR Y GUARDAR BASES
# =========================================

bases = {
    "RRHH": generar_rrhh(),
    "Ventas": generar_ventas(),
    "Finanzas": generar_finanzas(),
    "Produccion": generar_produccion(),
    "Marketing": generar_marketing()
}

for nombre, df in bases.items():
    df.to_csv(f"{nombre}.csv", index=False)

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from datetime import datetime

# ===============================
# CREAR CARPETA
# ===============================
if not os.path.exists("reportes_estadisticos"):
    os.makedirs("reportes_estadisticos")

# ===============================
# FUNCIÓN PARA CALCULAR MEDIDAS
# ===============================
def calcular_medidas(df):

    numericas = df.select_dtypes(include=np.number)

    resultados = []

    for col in numericas.columns:

        media = numericas[col].mean()
        mediana = numericas[col].median()
        moda = numericas[col].mode().iloc[0]
        varianza = numericas[col].var()
        desviacion = numericas[col].std()
        rango = numericas[col].max() - numericas[col].min()

        resultados.append({
            "Variable": col,
            "Media": round(media,2),
            "Mediana": round(mediana,2),
            "Moda": round(moda,2),
            "Varianza": round(varianza,2),
            "Desv. Estándar": round(desviacion,2),
            "Rango": round(rango,2)
        })

    return pd.DataFrame(resultados)


# ===============================
# FUNCIÓN PARA GENERAR GRÁFICOS
# ===============================
import base64
from io import BytesIO

def generar_graficos(df):

    numericas = df.select_dtypes(include=np.number)
    imagenes_base64 = []

    for col in numericas.columns:

        # Histograma
        fig, ax = plt.subplots()
        ax.hist(numericas[col], bins=30)
        ax.set_title(f"Histograma - {col}")
        ax.set_xlabel(col)
        ax.set_ylabel("Frecuencia")

        buffer = BytesIO()
        plt.savefig(buffer, format="png")
        buffer.seek(0)
        imagen_base64 = base64.b64encode(buffer.read()).decode('utf-8')
        imagenes_base64.append(imagen_base64)
        plt.close()

        # Boxplot
        fig, ax = plt.subplots()
        ax.boxplot(numericas[col])
        ax.set_title(f"Boxplot - {col}")

        buffer = BytesIO()
        plt.savefig(buffer, format="png")
        buffer.seek(0)
        imagen_base64 = base64.b64encode(buffer.read()).decode('utf-8')
        imagenes_base64.append(imagen_base64)
        plt.close()

    return imagenes_base64


# ===============================
# FUNCIÓN PARA GENERAR REPORTE
# ===============================
def generar_reporte(nombre_archivo):

    df = pd.read_csv(nombre_archivo)
    nombre_base = nombre_archivo.replace(".csv","")

    medidas = calcular_medidas(df)
    imagenes = generar_graficos(df)

    medidas_html = medidas.to_html(index=False)

    imagenes_html = ""
    for img in imagenes:
        imagenes_html += f'<img src="data:image/png;base64,{img}" width="600"><br><br>'

    reporte = f"""
    <html>
    <head>
        <title>Reporte Estadístico - {nombre_base}</title>
        <style>
            body {{
                font-family: Arial;
                margin: 40px;
                background-color: #f4f6f8;
            }}
            h1 {{
                color: #1f4e79;
            }}
            table {{
                border-collapse: collapse;
                width: 100%;
                background: white;
            }}
            th, td {{
                border: 1px solid #ddd;
                padding: 8px;
                text-align: center;
            }}
            th {{
                background-color: #1f4e79;
                color: white;
            }}
        </style>
    </head>
    <body>
        <h1>Reporte Estadístico Automático</h1>
        <h2>Base: {nombre_base}</h2>
        <h3>Fecha: {datetime.now()}</h3>

        <h2>Medidas de Tendencia Central y Dispersión</h2>
        {medidas_html}

        <h2>Visualizaciones</h2>
        {imagenes_html}
    </body>
    </html>
    """

    ruta_salida = f"reportes_estadisticos/{nombre_base}_reporte.html"
    with open(ruta_salida, "w", encoding="utf-8") as f:
        f.write(reporte)


# ===============================
# EJECUTAR PARA TODAS LAS BASES
# ===============================

archivos = [
    "RRHH.csv",
    "Ventas.csv",
    "Finanzas.csv",
    "Produccion.csv",
    "Marketing.csv"
]

for archivo in archivos:
    generar_reporte(archivo)

print("✅ Reportes estadísticos generados automáticamente.")

✅ Reportes estadísticos generados automáticamente.
